In [ ]:
import json
from pathlib import Path
import google.generativeai as genai
import os
from langchain_core.prompts import PromptTemplate
import time
from dotenv import load_dotenv

In [ ]:
prompt = PromptTemplate.from_template("""
    Task: You are an AI assistant that generates QA pairs from a given text extracted from a book. Generate specific question-and-answer pairs from the given input text.

    Input Text:
    {input}

    Instruction: Return a JSON object in the response that strictly matches the following structure:

    [
        {{
            "Question": "Example question 1",
            "Answer": "Example answer 1"
        }},
        {{
            "Question": "Example question 2",
            "Answer": "Example answer 2"
        }}
        // ... additional question-answer pairs ...
    ]

    Output:
""")

adaptorPrompt = PromptTemplate.from_template("""
    Task: You are an AI assistant that converts given text into correct json format.

                                                
    Input Text: 
    {input}
                                                                                                                              
    Instruction: Return a JSON object in the response that strictly matches the following structure:

        [
            {{
                "Question": "Example question 1",
                "Answer": "Example answer 1"
            }},
            {{
                "Question": "Example question 2",
                "Answer": "Example answer 2"
            }}
            // ... additional question-answer pairs ...
        ]
""")


In [ ]:
import re


def ollama_llm(prompt):
    response = ollama.chat(
        model="deepseek-r1",
        messages=[{"role": "user", "content": prompt}],
    )

    response_content = response["message"]["content"]

    # Remove content between <think> and </think> tags to remove thinking output
    final_answer = re.sub(r"<think>.*?</think>", "", response_content, flags=re.DOTALL).strip()

    return final_answer

In [ ]:
def generate_question_answer(chunk):
    formatted_prompt = prompt.format(input=chunk['input_text'])
    # print(f"{formatted_prompt}=")

    try:
        response = ollama_llm(formatted_prompt)
        print(f"{response=}")
        formatted_adapted_prompt = adaptorPrompt.format(input=response)
        chunk['qa_pairs'] = json.loads(response)
    except json.JSONDecodeError:
            while True:
                try:
                    time.sleep(5)
                    response = ollama_llm(formatted_adapted_prompt)
                    print(f"{response=}")
                    chunk['qa_pairs'] = json.loads(response)
                    break
                except json.JSONDecodeError:
                    continue
    return response 
    

In [ ]:
chunks_path = Path('booksChunks')
dataset_path = Path('Dataset')

for file in chunks_path.glob("*.json"):
    with open(file, 'r') as json_file:
        chunks_dict = json.load(json_file)
    count=0
    for chunk in chunks_dict:
        # Generate QA pairs and parse them as JSON
        chunk = generate_question_answer(chunk)
        print(chunk)
        print(f"Chunk {count} done")
        count+=1
        # if count==5:
        #     break
        
    # Save the updated chunks back to the file
    with open(os.path.join(dataset_path, os.path.basename(file)), 'w') as json_file:
        json.dump(chunks_dict, json_file, indent=2)
